# Register Workspace Assets in an External Azure ML Registry

Share existing registered models, environments, and data assets from your workspace into a separate registry. Source assets are not modified. The destination keeps the same asset names and uses explicit, immutable versions.

**Source:** Adapted from this repository's [asset-sharing notebook](../../../notebooks/asset_sharing/sharing_assets_registries_workspaces.ipynb). The model/environment sharing flow is retained; this notebook adds data sharing, workshop configuration, preview, collision checks, and registry readback.

Run this notebook from the AML compute instance, or another machine with private connectivity and DNS for the workspace, registry, and their Storage/ACR private endpoints. Laptop IP exceptions on supporting services do not expose the private AML APIs.

Complete notebooks 02-04 first. The caller needs workspace asset access and **AzureML Registry User** on the destination. Native sharing also requires the destination registry's resource-instance rule on source Storage. The new private Terraform stack configures these permissions and managed-network outbound rules.

This registers assets, not an online endpoint or a job. Sharing an environment can trigger an image build on the workspace image-build cluster. No image is built directly in the private registry.

In [1]:
from pathlib import Path
import os

from azure.ai.ml import MLClient
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from IPython.display import display
import pandas as pd

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

REGISTRY_NAME = os.environ["AZUREML_REGISTRY_NAME"].strip()
if not REGISTRY_NAME or REGISTRY_NAME.startswith("<"):
    raise ValueError("Set AZUREML_REGISTRY_NAME to the external registry name in workshop/.env")
SHARE = os.getenv("SHARE_FOUNDATION_ASSETS", "false").strip().lower() in {"1", "true", "yes"}
ASSET_TYPES = [value.strip().lower() for value in os.getenv("REGISTRY_ASSET_TYPES", "model,environment,data").split(",")]

credential = DefaultAzureCredential(
    managed_identity_client_id=os.getenv("DEFAULT_IDENTITY_CLIENT_ID") or None,
    exclude_environment_credential=True,
)
ml_client_workspace = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)
ml_client_registry = MLClient(credential=credential, registry_name=REGISTRY_NAME)
print(f"Source workspace: {os.environ['AZUREML_WORKSPACE_NAME']}; destination registry: {REGISTRY_NAME}")
print(f"Registration enabled: {SHARE}")

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Source workspace: mlwprivatencus06; destination registry: mlrprivatencus06
Registration enabled: True


## Select Assets and Versions

`REGISTRY_ASSET_TYPES` selects `model`, `environment`, and/or `data`. Names come from the existing `WORKSHOP_*_NAME` settings. Optional `REGISTRY_*_SOURCE_VERSION` settings select a particular workspace version; blank resolves the workspace's latest label once. Optional `REGISTRY_*_TARGET_VERSION` settings choose destination versions; blank preserves the resolved source version.

The notebook checks every destination before sharing anything. With registration enabled, any existing target version stops the operation instead of overwriting or silently reusing it. Choose a new target version or remove that type from the selection. Azure operations are not transactional: if a later service operation fails, already completed shares remain and their errors are not suppressed.

In [2]:
def resolve_workspace_assets(
    workspace_client: MLClient,
    asset_types: list[str],
    settings: dict[str, str],
) -> list[dict[str, str]]:
    collections = {"model": "models", "environment": "environments", "data": "data"}
    if not asset_types or len(set(asset_types)) != len(asset_types):
        raise ValueError("Select one or more distinct asset types: model, environment, data")
    if any(kind not in collections for kind in asset_types):
        raise ValueError("Supported asset types are model, environment, and data")

    assets = []
    for kind in asset_types:
        name = settings.get(f"WORKSHOP_{kind.upper()}_NAME", "").strip()
        if not name or name.startswith("<"):
            raise ValueError(f"Set WORKSHOP_{kind.upper()}_NAME to an existing workspace asset")
        operations = {"model": workspace_client.models, "environment": workspace_client.environments, "data": workspace_client.data}[kind]
        source_version = settings.get(f"REGISTRY_{kind.upper()}_SOURCE_VERSION", "").strip()
        source = (
            operations.get(name, version=source_version)
            if source_version else operations.get(name, label="latest")
        )
        if not source.version or not source.id:
            raise ValueError(f"Workspace {kind} {name} has no registered version or resource ID")
        target_version = settings.get(f"REGISTRY_{kind.upper()}_TARGET_VERSION", "").strip() or source.version
        assets.append({
            "asset_type": kind,
            "collection": collections[kind],
            "name": name,
            "source_version": source.version,
            "target_version": target_version,
            "source_id": source.id,
        })
    return assets


def share_workspace_assets(
    workspace_client: MLClient,
    registry_client: MLClient,
    registry_name: str,
    assets: list[dict[str, str]],
    enabled: bool,
) -> list[dict[str, str]]:
    if not assets:
        raise ValueError("No assets were selected for sharing")
    source_operations = {"models": workspace_client.models, "environments": workspace_client.environments, "data": workspace_client.data}
    target_operations = {"models": registry_client.models, "environments": registry_client.environments, "data": registry_client.data}
    results = []
    for asset in assets:
        operations = target_operations[asset["collection"]]
        try:
            existing = operations.get(asset["name"], version=asset["target_version"])
        except ResourceNotFoundError:
            results.append({**asset, "status": "ready", "id": ""})
        else:
            if not existing.id:
                raise RuntimeError(f"Registry readback has no resource ID for {asset['name']}:{asset['target_version']}")
            results.append({**asset, "status": "exists", "id": existing.id})
    if not enabled:
        return results
    if any(result["status"] == "exists" for result in results):
        raise ValueError("Destination asset versions already exist. Choose new REGISTRY_*_TARGET_VERSION values; no assets were shared.")

    for result in results:
        operations = source_operations[result["collection"]]
        operations.share(
            name=result["name"],
            version=result["source_version"],
            registry_name=registry_name,
            share_with_name=result["name"],
            share_with_version=result["target_version"],
        )
        verified = target_operations[result["collection"]].get(
            result["name"], version=result["target_version"]
        )
        if verified.name != result["name"] or verified.version != result["target_version"] or not verified.id:
            raise RuntimeError(f"Registry readback did not match {result['name']}:{result['target_version']}")
        result["status"] = "registered"
        result["id"] = verified.id
    return results

In [3]:
asset_plan = resolve_workspace_assets(ml_client_workspace, ASSET_TYPES, dict(os.environ))
display(pd.DataFrame(asset_plan)[["asset_type", "name", "source_version", "target_version", "source_id"]])

,asset_type,name,source_version,target_version,source_id
0,model,workshop-taxi-model,1,1,/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70f...
1,environment,workshop-taxi-environment,1,1,/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70f...
2,data,workshop-taxi-data,1,1,/subscriptions/5784b6a5-de3f-4fa4-8b8f-e5bb70f...


## Preview or Register

Leave `SHARE_FOUNDATION_ASSETS=false` to check destination versions without changing Azure. To register the displayed versions, set it to `true` in `workshop/.env`, rerun the configuration cell and the cell below, and then return it to `false`. Rerunning configuration does not replace the already resolved source-version plan.

For a private registry, use SDK/CLI readback rather than relying on Studio to display all asset types. If sharing is denied by source Storage, verify the registry resource-instance rule and selected-network firewall configuration. Do not enable all-network access, account keys, or ACR admin credentials as a workaround.

In [4]:
results = share_workspace_assets(
    ml_client_workspace, ml_client_registry, REGISTRY_NAME, asset_plan, enabled=SHARE
)
display(pd.DataFrame(results)[["asset_type", "name", "source_version", "target_version", "status", "id"]])
if not SHARE:
    print("Preview only. Set SHARE_FOUNDATION_ASSETS=true in workshop/.env and rerun configuration before registering.")
else:
    print("Every selected asset was registered and retrieved from the external registry. Reset SHARE_FOUNDATION_ASSETS=false.")

Method share: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


,asset_type,name,source_version,target_version,status,id
0,model,workshop-taxi-model,1,1,registered,azureml://registries/mlrprivatencus06/models/w...
1,environment,workshop-taxi-environment,1,1,registered,azureml://registries/mlrprivatencus06/environm...
2,data,workshop-taxi-data,1,1,registered,azureml://registries/mlrprivatencus06/data/wor...


Every selected asset was registered and retrieved from the external registry. Reset SHARE_FOUNDATION_ASSETS=false.
